# Code mẫu — Logistic và softmax regression

Logistic regression biến một điểm tuyến tính thành xác suất cho bài toán hai lớp. Softmax làm việc tương tự với nhiều điểm, cho bài toán nhiều lớp. Các ví dụ dưới đây kiểm cả phép tính sao cho không tràn số, lẫn cách ra quyết định từ xác suất sau khi model đã fit.

## Ví dụ 1: Tăng trọng số để sửa một email bị đoán thấp

Một email có `x = 2` đường link, và đúng là spam, `y = 1`. Model hiện có `w = 1`, `b = −1`, nên logit bằng 1. Ta giữ nguyên bias, chỉ xem một bước cập nhật trọng số làm thay đổi những gì.

In [ ]:
import numpy as np

def sigmoid(z):
    # Không tính thẳng exp(-z), vì khi z rất âm thì exp(-z) sẽ tràn số.
    return np.exp(-np.logaddexp(0., -np.asarray(z)))

x, target = 2., 1.
w, bias = 1., -1.
logit = w * x + bias
probability = float(sigmoid(logit))
loss = float(np.logaddexp(0., logit) - target * logit)
gradient_w = (probability - target) * x
new_w = w - 0.5 * gradient_w
new_logit = new_w * x + bias
print("logit / xác suất / loss:", round(logit, 4), round(probability, 4), round(loss, 4))
print("gradient w:", round(gradient_w, 4), "w mới:", round(new_w, 4))
print("xác suất / loss mới:", round(float(sigmoid(new_logit)), 4),
      round(float(np.logaddexp(0., new_logit) - target * new_logit), 4))
extreme = np.array([-1000., 0., 1000.])
print("sigmoid ở logits lớn:", sigmoid(extreme).round(3).tolist())
assert gradient_w < 0 and float(sigmoid(new_logit)) > probability

logit / xác suất / loss: 1.0 0.7311 0.3133
gradient w: -0.5379 w mới: 1.2689
xác suất / loss mới: 0.8232 0.1946
sigmoid ở logits lớn: [0.0, 0.5, 1.0]


Gradient theo `w` bằng `(p − y) × x`. Ở đây `p = 0.73` còn thấp hơn nhãn 1, nên gradient âm, và trừ đi gradient âm thì `w` tăng. Sau một bước, xác suất spam tăng từ 0.73 lên 0.82, còn loss giảm từ 0.31 xuống 0.19. Đây là cập nhật cho đúng một mẫu, và giữ bias đứng yên chỉ để dễ quan sát. Khi train thật, gradient được lấy trên cả tập dữ liệu, và mọi tham số đều được cập nhật. Dòng cuối cho thấy cách viết bằng `logaddexp` vẫn ra kết quả hữu hạn ở logit ±1000, nơi cách tính sigmoid thẳng dễ bị tràn số.

## Ví dụ 2: Cùng một xác suất, đổi ngưỡng chặn thư

Mỗi email được mô tả bằng số đường link và số từ quảng cáo. Mười hai email làm train, sáu email khác có nhãn làm validation. Giả sử chặn nhầm một thư thật thì thiệt hại là 5, còn để lọt một thư spam thì thiệt hại là 1. Khi hai loại lỗi có giá khác nhau như vậy, accuracy chưa đủ để chọn ngưỡng.

In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression

X_train = np.array([[0, 0], [0, 1], [1, 0], [1, 1], [2, 0], [2, 1],
                    [1, 3], [2, 2], [3, 1], [3, 3], [4, 2], [4, 4]], dtype=float)
y_train = np.array([0, 0, 0, 0, 0, 1, 0, 1, 1, 1, 1, 1])
X_valid = np.array([[0, 2], [1, 2], [2, 1], [2, 3], [3, 0], [4, 3]], dtype=float)
y_valid = np.array([0, 0, 1, 1, 0, 1])
model = LogisticRegression(C=1., max_iter=500).fit(X_train, y_train)
spam_column = int(np.flatnonzero(model.classes_ == 1)[0])
probabilities = model.predict_proba(X_valid)[:, spam_column]
print("classes_:", model.classes_.tolist())
print("(nhãn, P(spam)):", list(zip(y_valid.tolist(), probabilities.round(3).tolist())))
for threshold in [0.3, 0.5, 0.8]:
    blocked = probabilities >= threshold
    false_positive = int(np.sum(blocked & (y_valid == 0)))
    false_negative = int(np.sum(~blocked & (y_valid == 1)))
    print("ngưỡng", threshold, "chặn nhầm:", false_positive,
          "lọt spam:", false_negative, "chi phí:", 5 * false_positive + false_negative)

classes_: [0, 1]
(nhãn, P(spam)): [(0, 0.095), (0, 0.291), (1, 0.503), (1, 0.715), (0, 0.714), (1, 0.974)]
ngưỡng 0.3 chặn nhầm: 1 lọt spam: 0 chi phí: 5
ngưỡng 0.5 chặn nhầm: 1 lọt spam: 0 chi phí: 5
ngưỡng 0.8 chặn nhầm: 0 lọt spam: 2 chi phí: 2


Code lấy cột xác suất spam qua `classes_`, chứ không mặc định là cột thứ hai. Đây là thói quen nên giữ, nhất là khi nhãn không phải 0 và 1. Ở ngưỡng 0.8, hai thư spam bị lọt, nhưng không thư thật nào bị chặn nhầm, nên chi phí chỉ là 2, thấp hơn mức 5 ở ngưỡng 0.5. Đổi ngưỡng chỉ đổi quyết định, chứ không fit lại hệ số nào. Nếu dùng bảng này để chọn ngưỡng, thì sáu email ấy đã là validation, không còn là test độc lập. Khi xác suất được hiệu chuẩn tốt, ngưỡng tối ưu còn có thể tính thẳng từ hai mức chi phí. Nhưng một model nhỏ như thế này chưa chứng minh được xác suất của nó đã hiệu chuẩn tốt.

## Ví dụ 3: Ba chủ đề với logits rất lớn

Hai hàng logits giả lập ứng với ba chủ đề `thể thao, khoa học, âm nhạc`. Chúng chỉ dùng để kiểm cách tính softmax, chứ không phải output của một model ngôn ngữ nào. Ta muốn thấy rằng cộng 1000 vào mọi điểm của một hàng thì xác suất không đổi.

In [ ]:
import numpy as np

def log_softmax(logits):
    shifted = logits - logits.max(axis=1, keepdims=True)
    return shifted - np.log(np.exp(shifted).sum(axis=1, keepdims=True))

logits = np.array([[2., 1., 0.], [0., 1., 2.]])
labels = np.array([1, 2])  # Chỉ số của lớp đúng ở từng văn bản.
log_prob = log_softmax(logits)
prob = np.exp(log_prob)
print("xác suất:", prob.round(4).tolist())
print("tổng từng hàng:", prob.sum(axis=1).round(4).tolist())
print("cộng 1000 không đổi:", np.allclose(log_prob, log_softmax(logits + 1000)))
print("loss từng mẫu:", (-log_prob[np.arange(len(labels)), labels]).round(4).tolist())
one_hot = np.eye(3)[labels]
gradient = (prob - one_hot) / len(labels)  # Gradient của cross-entropy trung bình theo logits.
print("gradient logits:", gradient.round(4).tolist())
print("tổng gradient mỗi hàng:", gradient.sum(axis=1).round(6).tolist())
assert np.allclose(prob.sum(axis=1), 1.)
assert np.allclose(gradient.sum(axis=1), 0.)

xác suất: [[0.6652, 0.2447, 0.09], [0.09, 0.2447, 0.6652]]
tổng từng hàng: [1.0, 1.0]
cộng 1000 không đổi: True
loss từng mẫu: [1.4076, 0.4076]
gradient logits: [[0.3326, -0.3776, 0.045], [0.045, 0.1224, -0.1674]]
tổng gradient mỗi hàng: [-0.0, -0.0]


Mẹo trừ đi giá trị lớn nhất của hàng trước khi lấy `exp` giúp không bao giờ tràn số, và không làm đổi kết quả. Dòng “cộng 1000 không đổi” kiểm đúng điều ấy. Ở hàng đầu, lớp đúng là khoa học, dù thể thao đang có điểm cao nhất, nên loss của mẫu này lớn hơn mẫu thứ hai. Gradient của lớp khoa học âm, nên một bước đi ngược gradient sẽ tăng điểm lớp đúng, còn hai lớp kia bị giảm. Tổng gradient mỗi hàng bằng 0, khớp với chuyện cộng cùng một hằng số vào cả hàng không làm đổi softmax. Tính thẳng log-softmax như vậy còn giữ loss hữu hạn, kể cả khi một xác suất nhỏ tới mức bị làm tròn thành 0.

## Ví dụ 4: Từ số đo bông hoa tới xác suất của ba loài

Iris là bộ dữ liệu có sẵn trong scikit-learn, gồm bốn kích thước đo bằng cm và ba loài hoa. Ta chia stratified để mỗi tập có đủ cả ba loài, rồi fit scaler và logistic regression trong cùng một Pipeline. Ngoài accuracy, ta đọc ba hàng xác suất đầu, và kiểm các cột xác suất theo thứ tự của `classes_`.

In [ ]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

data = load_iris()
X_train, X_test, y_train, y_test = train_test_split(
    data.data, data.target, test_size=0.3, stratify=data.target, random_state=8
)
pipeline = make_pipeline(StandardScaler(), LogisticRegression(max_iter=500))
pipeline.fit(X_train, y_train)
classifier = pipeline[-1]
prob = pipeline.predict_proba(X_test)
chosen_labels = classifier.classes_[np.argmax(prob, axis=1)]
print("cột xác suất:", data.target_names[classifier.classes_].tolist())
print("3 hàng xác suất:", prob[:3].round(3).tolist())
print("3 nhãn thật / đoán:", y_test[:3].tolist(), chosen_labels[:3].tolist())
print("accuracy / log loss:", round(accuracy_score(y_test, chosen_labels), 3),
      round(log_loss(y_test, prob, labels=classifier.classes_), 3))
print("shape / tổng hàng hợp lệ:", prob.shape, np.allclose(prob.sum(axis=1), 1.))
assert np.array_equal(chosen_labels, pipeline.predict(X_test))

cột xác suất: ['setosa', 'versicolor', 'virginica']
3 hàng xác suất: [[0.021, 0.874, 0.105], [0.0, 0.04, 0.96], [0.0, 0.125, 0.875]]
3 nhãn thật / đoán: [1, 2, 2] [1, 2, 2]
accuracy / log loss: 0.889 0.236
shape / tổng hàng hợp lệ: (45, 3) True


Accuracy 0.889 nghĩa là đúng 40 trong 45 bông. Ba hàng đầu đều đúng, nhưng chúng không đại diện cho mọi hàng của test. Hàng đầu dành 0.874 cho versicolor, tức cột thứ hai theo thứ tự `classes_`. Log loss khác accuracy ở chỗ nó còn xét model tự tin vào lớp thật tới mức nào, chứ không chỉ xét đúng hay sai. Với ba lớp và solver mặc định L-BFGS, scikit-learn tối ưu thẳng một model softmax, chứ không ghép nhiều model hai lớp. Scaler chỉ học từ train, và mọi con số ở đây gắn với một lần chia cố định.